# Provision the LEX project

This notebook does three things: (1) create LEX's project schema and
register it in the shared catalog, (2) create LEX's contract-specific
tables (CONTRACT_REGISTER / CONTRACT_DOCUMENT_LINK / CONTRACT_FIELD_EXTRACTS)
and access role, and (3) deploy/redeploy the Streamlit-in-Snowflake app. It
intentionally does not sync, ingest, or index documents, or link/extract
contract data — that all happens later, on demand, from the app itself,
starting with uploading the Required Contracts Register workbook on the
Data Sources page.

Forked from the `project-llm-wiki` provisioning notebook running in
production for `ORG_MM_CHAT` (`ds-madhavan-ramani/org_mm_chat`). Run top to
bottom the first time; most cells are safe to re-run any time after that
(each says so, or not, in its own markdown).

In [ ]:
import sys
sys.path.insert(0, '../python')
from snowflake_session import get_session

session = get_session()
session.sql("USE ROLE ADVANCEDANALYTICS").collect()
session.sql("USE WAREHOUSE MTMWH02").collect()
session.sql("USE DATABASE MEDSOCMS").collect()
print('Connected.')

## One-time only: create the shared catalog schema
Skip this cell if `MEDSOCMS.APP_CATALOG` already exists — it's the same
shared catalog every project-llm-wiki project registers in, including
`ORG_MM_CHAT` in the sibling repo, so this may well already exist on this
account. The `CREATE ... IF NOT EXISTS` statements make re-running this
harmless either way.

In [ ]:
from utils.sql_script import run_sql_file

count = run_sql_file(session, '../sql/00_setup_catalog.sql')
print(f'Catalog schema ready ({count} statements executed).')

## Create LEX's dedicated database + compute pool
LEX asks for full isolation — its own database (`MEDSCOMA`), not just its
own schema inside the shared `MEDSOCMS` database — plus its own compute
pool for container-runtime Streamlit (needed for a modern, pinned Streamlit
version and predictable resources against 100-500 page OCR/indexing runs).

Both `CREATE DATABASE` and `CREATE COMPUTE POOL` are typically
`SYSADMIN`/`ACCOUNTADMIN`-only privileges that `ADVANCEDANALYTICS` doesn't
hold by default (the same gotcha `ORG_MM_CHAT`'s README documents for
compute pools). This cell tries anyway and prints clear next steps if it
can't — if it fails, ask whoever holds `SYSADMIN` to run the two
`CREATE ...` statements printed in the error output, then re-run this cell
to confirm (it'll report "already exists" and move on). During build,
`QUERY_WAREHOUSE` stays `MTMWH02` (the shared build warehouse) — swap it to
a dedicated production warehouse later via `ALTER` on the project's catalog
row before productionisation.

In [ ]:
LEX_DATABASE = 'MEDSCOMA'
LEX_COMPUTE_POOL_NAME = 'STREAMLIT_COMPUTE_POOL_CONTRACT_MGMT'

def _try_create(sql_stmt, what):
    try:
        session.sql(sql_stmt).collect()
        print(f'OK  {what}')
    except Exception as e:
        print(f'SKIPPED — could not create {what} with the current role.')
        print(f'  Ask someone with SYSADMIN (or ACCOUNTADMIN) to run:')
        print(f'    {sql_stmt}')
        print(f'  Original error: {e}')

_try_create(
    f"""CREATE DATABASE IF NOT EXISTS {LEX_DATABASE}
        COMMENT = 'Dedicated database for LEX / Transition Contract Management RAG project'""",
    f'database {LEX_DATABASE}',
)
session.sql('USE WAREHOUSE MTMWH02').collect()
_try_create(
    f"""CREATE COMPUTE POOL IF NOT EXISTS {LEX_COMPUTE_POOL_NAME}
        MIN_NODES = 1 MAX_NODES = 2
        INSTANCE_FAMILY = CPU_X64_XS
        AUTO_SUSPEND_SECS = 300
        AUTO_RESUME = TRUE
        COMMENT = 'Dedicated compute pool for the LEX / Transition Contract Management Streamlit app'""",
    f'compute pool {LEX_COMPUTE_POOL_NAME}',
)

## Create the LEX project
Fill in the values below and run. Skip this cell if the project already
exists (re-running errors on the duplicate project_code) — the `UPDATE`
in the same cell still runs either way, so editing a setting below and
re-running always takes effect even after the project exists.

**Confirm before running for real** (see the plan's open questions):
`SHAREPOINT_SITE_URL` / `SHAREPOINT_DEFAULT_FOLDER` (is the "network
drive" actually a SharePoint library, and which folder), and
`CREATED_BY`. Left blank below on purpose rather than guessed.

In [ ]:
PROJECT_CODE = 'LEX'
PROJECT_NAME = 'LEX - Legal EXtraction & Contract Intelligence'
DESCRIPTION = 'Contract Q&A and automated stock-field extraction for Signed & Executed Contracts (MR5 Transition Contracts Team)'
SHAREPOINT_SITE_URL = ''            # TODO confirm: the contracts library's SharePoint site
SHAREPOINT_DEFAULT_FOLDER = ''      # TODO confirm: the contracts library's default folder URL
CREATED_BY = ''                     # TODO: your name
QUERY_WAREHOUSE = 'MTMWH02'
COMPUTE_POOL = LEX_COMPUTE_POOL_NAME   # container runtime, unlike ORG_MM_CHAT's warehouse-runtime default
DATA_DATABASE = LEX_DATABASE           # LEX's own database, not the shared MEDSOCMS
SEGMENTATION_PROFILE = 'LEX_CONTRACT'  # clause/schedule-aware; see index_builder.py

# Retrieval mechanism config — see the sibling ORG_MM_CHAT README's "How
# retrieval works" for what each one does/costs. Unlike the
# CALL CREATE_PROJECT(...) below (first-creation only), the UPDATE that
# applies these runs every time this cell runs — edit a value and re-run
# to change an already-existing project's settings.
SEGMENTATION_GRANULARITY = 'DETAILED'  # clause-level sections, not one per whole contract
ENABLE_RERANKING = True
ENABLE_VECTOR_SEARCH = True    # on from day one — summary-only routing is too coarse once
                                 # this project scales toward ~600 contracts
MAX_CANDIDATE_DOCS = 10
# Doubles as the per-chunk size for documents longer than this (see
# index_builder.py's chunked indexing) — sized well within typical Cortex
# model input limits per call, unlike a single-shot cap large enough to
# hold a whole 500-page contract, which would risk exceeding those limits
# outright rather than just chunking cleanly.
MAX_DOCUMENT_CHARS = 100000

existing = session.sql(
    "SELECT COUNT(*) AS C FROM MEDSOCMS.APP_CATALOG.PROJECTS WHERE PROJECT_CODE = ?",
    params=[PROJECT_CODE],
).collect()[0]["C"]

if existing > 0:
    print(f"Project '{PROJECT_CODE}' already exists — updating its settings below.")
else:
    result = session.sql(
        'CALL CREATE_PROJECT(?, ?, ?, ?, ?, ?, ?, ?, ?)',
        params=[PROJECT_CODE, PROJECT_NAME, DESCRIPTION, SHAREPOINT_SITE_URL,
                SHAREPOINT_DEFAULT_FOLDER, CREATED_BY, QUERY_WAREHOUSE, COMPUTE_POOL,
                DATA_DATABASE],
    ).collect()
    print(result[0][0])

session.sql(
    """UPDATE MEDSOCMS.APP_CATALOG.PROJECTS
       SET SEGMENTATION_PROFILE = ?, SEGMENTATION_GRANULARITY = ?,
           ENABLE_RERANKING = ?, ENABLE_VECTOR_SEARCH = ?, MAX_CANDIDATE_DOCS = ?,
           MAX_DOCUMENT_CHARS = ?
       WHERE PROJECT_CODE = ?""",
    params=[SEGMENTATION_PROFILE, SEGMENTATION_GRANULARITY,
            ENABLE_RERANKING, ENABLE_VECTOR_SEARCH, MAX_CANDIDATE_DOCS,
            MAX_DOCUMENT_CHARS, PROJECT_CODE],
).collect()
print(f"Settings applied: profile={SEGMENTATION_PROFILE}, granularity={SEGMENTATION_GRANULARITY}, "
      f"reranking={ENABLE_RERANKING}, vector_search={ENABLE_VECTOR_SEARCH}, "
      f"max_docs={MAX_CANDIDATE_DOCS}, max_document_chars={MAX_DOCUMENT_CHARS}")

### Optional: point LEX at its own dedicated Graph API app registration
Leave this cell as-is (does nothing) to keep using the shared tenant-level
Graph app every project-llm-wiki project uses by default. Fill in the three
values and re-run once a dedicated, least-privilege app registration for
LEX exists (see `sql/test_graph_connectivity.sql`'s bottom section for the
one-time `CREATE SECRET`/`ALTER EXTERNAL ACCESS INTEGRATION` steps that
have to happen first).

In [ ]:
LEX_GRAPH_TENANT_ID = None   # e.g. 'xxxxxxxx-xxxx-...'
LEX_GRAPH_CLIENT_ID = None
LEX_GRAPH_SECRET_NAME = None  # e.g. 'MEDSOCMS.APP_CATALOG.LEX_GRAPH_API_SECRET'

if LEX_GRAPH_TENANT_ID and LEX_GRAPH_CLIENT_ID and LEX_GRAPH_SECRET_NAME:
    session.sql(
        """UPDATE MEDSOCMS.APP_CATALOG.PROJECTS
           SET GRAPH_TENANT_ID = ?, GRAPH_CLIENT_ID = ?, GRAPH_SECRET_NAME = ?
           WHERE PROJECT_CODE = ?""",
        params=[LEX_GRAPH_TENANT_ID, LEX_GRAPH_CLIENT_ID, LEX_GRAPH_SECRET_NAME, PROJECT_CODE],
    ).collect()
    print('Dedicated Graph app registration set for LEX.')
else:
    print('No dedicated registration configured — LEX will use the shared tenant-level Graph app.')

## Create LEX's contract tables
`CONTRACT_REGISTER`, `CONTRACT_DOCUMENT_LINK`, and `CONTRACT_FIELD_EXTRACTS`
are LEX-specific — not part of the generic project-llm-wiki shape
`CREATE_PROJECT` already created (`RAW_DOCUMENTS`/`DOCUMENT_INDEX`). See
`sql/03_lex_contract_tables.sql` for the reference copy of this DDL. Safe
to re-run any time (idempotent `CREATE TABLE IF NOT EXISTS`).

In [ ]:
proj_row = session.sql(
    "SELECT DATA_DATABASE, DATA_SCHEMA FROM MEDSOCMS.APP_CATALOG.PROJECTS WHERE PROJECT_CODE = ?",
    params=[PROJECT_CODE],
).collect()
if not proj_row:
    raise ValueError(f"No project found with code '{PROJECT_CODE}' — run the project-creation cell first.")
qualified_schema = f"{proj_row[0]['DATA_DATABASE']}.{proj_row[0]['DATA_SCHEMA']}"

contract_table_ddl = [
    f"""CREATE TABLE IF NOT EXISTS {qualified_schema}.CONTRACT_REGISTER (
          CONTRACT_ID       INT IDENTITY PRIMARY KEY,
          CW_NUMBER          VARCHAR(50) NOT NULL UNIQUE,
          CONTRACT_TITLE      VARCHAR(500),
          STATUS               VARCHAR(20) DEFAULT 'ACTIVE',
          OVERVIEW_SUMMARY      VARCHAR(4000),
          OVERVIEW_GENERATED_AT  TIMESTAMP_NTZ,
          RECOMMENDED_ACTIONS        VARIANT,
          CLASSIFICATION_SCORECARD    VARIANT,
          CREATED_AT                    TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
        )""",
    f"""CREATE TABLE IF NOT EXISTS {qualified_schema}.CONTRACT_DOCUMENT_LINK (
          LINK_ID          INT IDENTITY PRIMARY KEY,
          CONTRACT_ID       INT NOT NULL REFERENCES {qualified_schema}.CONTRACT_REGISTER(CONTRACT_ID),
          DOC_ID             INT NOT NULL REFERENCES {qualified_schema}.RAW_DOCUMENTS(DOC_ID),
          DOC_ROLE            VARCHAR(20) NOT NULL,
          EFFECTIVE_DATE       DATE,
          SEQUENCE_NO           INT,
          LINKED_BY              VARCHAR(200),
          LINKED_AT               TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
          UNIQUE (CONTRACT_ID, DOC_ID)
        )""",
    f"""CREATE TABLE IF NOT EXISTS {qualified_schema}.CONTRACT_FIELD_EXTRACTS (
          EXTRACT_ID        INT IDENTITY PRIMARY KEY,
          CONTRACT_ID        INT NOT NULL REFERENCES {qualified_schema}.CONTRACT_REGISTER(CONTRACT_ID),
          FIELD_KEY           VARCHAR(50) NOT NULL,
          FIELD_VALUE           VARCHAR(4000),
          SOURCE_DOC_ID           INT REFERENCES {qualified_schema}.RAW_DOCUMENTS(DOC_ID),
          SOURCE_NODE_ID           INT REFERENCES {qualified_schema}.DOCUMENT_INDEX(NODE_ID),
          SOURCE_QUOTE               VARCHAR(4000),
          HIGHLIGHT_PHRASE            VARCHAR(500),
          CONFIDENCE                   VARCHAR(20),
          MODEL_USED                    VARCHAR(50),
          EXTRACTED_AT                   TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
          IS_VERIFIED                     BOOLEAN DEFAULT FALSE,
          VERIFIED_BY                      VARCHAR(200),
          VERIFIED_AT                       TIMESTAMP_NTZ,
          UNIQUE (CONTRACT_ID, FIELD_KEY)
        )""",
]
for stmt in contract_table_ddl:
    session.sql(stmt).collect()
print(f'Contract tables ready in {qualified_schema}.')

## Restrict access: the `LEX_USERS` role
Access control lives entirely at the Snowflake grant, not in application
code — `Chat.py` has no login screen. `LEX_USERS` gets `USAGE` on the
Streamlit app and its compute pool; add the 3-5 named team members' actual
usernames to `NAMED_USERS` below and re-run any time membership changes
(re-granting an already-granted role is a harmless no-op).

In [ ]:
NAMED_USERS = [
    # 'JSMITH',
    # 'APATEL',
]

app_row = session.sql(
    "SELECT STREAMLIT_APP_NAME, COMPUTE_POOL, QUERY_WAREHOUSE, DATA_DATABASE, DATA_SCHEMA "
    "FROM MEDSOCMS.APP_CATALOG.PROJECTS WHERE PROJECT_CODE = ?",
    params=[PROJECT_CODE],
).collect()[0]

session.sql("CREATE ROLE IF NOT EXISTS LEX_USERS").collect()
session.sql(
    f"GRANT USAGE ON STREAMLIT MEDSOCMS.APP_CATALOG.{app_row['STREAMLIT_APP_NAME']} TO ROLE LEX_USERS"
).collect()
if app_row["COMPUTE_POOL"]:
    session.sql(f"GRANT USAGE ON COMPUTE POOL {app_row['COMPUTE_POOL']} TO ROLE LEX_USERS").collect()
session.sql(f"GRANT USAGE ON WAREHOUSE {app_row['QUERY_WAREHOUSE']} TO ROLE LEX_USERS").collect()
session.sql(f"GRANT USAGE ON DATABASE {app_row['DATA_DATABASE']} TO ROLE LEX_USERS").collect()
session.sql(f"GRANT USAGE ON SCHEMA {app_row['DATA_DATABASE']}.{app_row['DATA_SCHEMA']} TO ROLE LEX_USERS").collect()

for user in NAMED_USERS:
    session.sql(f"GRANT ROLE LEX_USERS TO USER {user}").collect()
    print(f'Granted LEX_USERS to {user}')

if not NAMED_USERS:
    print('LEX_USERS role + object grants are in place. Add usernames to NAMED_USERS and re-run to grant it to people.')

## Deploy the Streamlit app
Safe to re-run any time — re-stages every file with `overwrite=True` and
redeploys the app object with `CREATE OR REPLACE`. `environment.yml` is
staged at the stage **root** (not nested), since Streamlit-in-Snowflake
only reads it from there.

In [ ]:
import os

proj = session.sql(
    "SELECT * FROM MEDSOCMS.APP_CATALOG.PROJECTS WHERE PROJECT_CODE = ?",
    params=[PROJECT_CODE],
).collect()
if not proj:
    raise ValueError(f"No project found with code '{PROJECT_CODE}' — run the project-creation cell first.")
p = proj[0]

STREAMLIT_STAGE = f"MEDSOCMS.APP_CATALOG.{p['STREAMLIT_STAGE_NAME']}"
STREAMLIT_APP_NAME = f"MEDSOCMS.APP_CATALOG.{p['STREAMLIT_APP_NAME']}"
APP_QUERY_WAREHOUSE = p['QUERY_WAREHOUSE']
APP_GRAPH_SECRET = p['GRAPH_SECRET_NAME'] or 'MEDSOCMS.APP_CATALOG.GRAPH_API_SECRET'

APP_COMPUTE_POOL = p['COMPUTE_POOL']
if not APP_COMPUTE_POOL or str(APP_COMPUTE_POOL).strip().lower() in ("", "none", "null"):
    APP_COMPUTE_POOL = None

session.sql(f"CREATE STAGE IF NOT EXISTS {STREAMLIT_STAGE}").collect()

# python/ keeps its own folder structure (python/config.py -> @stage/python/config.py),
# a sibling of Chat.py at the stage root. streamlit/'s contents are staged
# FLATTENED to the stage root (streamlit/Chat.py -> @stage/Chat.py,
# streamlit/pages/1_Data_Sources.py -> @stage/pages/1_Data_Sources.py) — a
# nested MAIN_FILE reliably fails to load on this account.
for root, _, files in os.walk("../python"):
    rel_root = os.path.relpath(root, "..")
    stage_dir = f"@{STREAMLIT_STAGE}/{rel_root}"
    for fname in files:
        if fname.endswith(".py"):
            session.file.put(f"{root}/{fname}", stage_dir,
                              auto_compress=False, overwrite=True)

for root, _, files in os.walk("../streamlit"):
    rel_root = os.path.relpath(root, "../streamlit")   # "." or "pages"
    stage_dir = f"@{STREAMLIT_STAGE}" if rel_root == "." else f"@{STREAMLIT_STAGE}/{rel_root}"
    for fname in files:
        if fname.endswith(".py"):
            session.file.put(f"{root}/{fname}", stage_dir,
                              auto_compress=False, overwrite=True)

# assets/ (the Contract Workspace Summary Template .docx) is staged as its
# own top-level folder too, a sibling of python/ — docx_report.py resolves
# it via os.path.dirname(__file__)/../assets/..., which on the stage means
# @stage/python/../assets/... = @stage/assets/..., so this folder name and
# nesting level both matter, not just "getting the file onto the stage
# somewhere".
for root, _, files in os.walk("../assets"):
    rel_root = os.path.relpath(root, "..")
    stage_dir = f"@{STREAMLIT_STAGE}/{rel_root}"
    for fname in files:
        session.file.put(f"{root}/{fname}", stage_dir,
                          auto_compress=False, overwrite=True)

# Stage EXACTLY ONE dependency manifest, matching the runtime this project is
# actually configured for. Staging both at once is ambiguous.
if APP_COMPUTE_POOL:
    manifest_to_stage = "pyproject.toml"
    manifest_to_remove = "environment.yml"
else:
    manifest_to_stage = "environment.yml"
    manifest_to_remove = "pyproject.toml"

local_path = f"../streamlit/{manifest_to_stage}"
if os.path.exists(local_path):
    session.file.put(local_path, f"@{STREAMLIT_STAGE}",
                      auto_compress=False, overwrite=True)

session.sql(f"REMOVE @{STREAMLIT_STAGE}/{manifest_to_remove}").collect()

# FROM (not the legacy ROOT_LOCATION) — required on accounts where
# ROOT_LOCATION has been retired for new/replaced Streamlit apps.
create_stmt = f"""
    CREATE OR REPLACE STREAMLIT {STREAMLIT_APP_NAME}
      FROM '@{STREAMLIT_STAGE}'
      MAIN_FILE = 'Chat.py'
      QUERY_WAREHOUSE = {APP_QUERY_WAREHOUSE}
"""
# GRAPH_API_ACCESS_INTEGRATION is required on every runtime for the Data
# Sources page's outbound Graph API calls. SECRETS binds the resolved
# Graph API client secret (LEX's own dedicated one if configured above,
# else the shared tenant-level default) under the local alias
# 'graph_secret' — utils/graph_client.py always reads that fixed alias
# regardless of which underlying secret object it points to.
if APP_COMPUTE_POOL:
    create_stmt += f"""
      RUNTIME_NAME = 'SYSTEM$ST_CONTAINER_RUNTIME_PY3_11'
      COMPUTE_POOL = '{APP_COMPUTE_POOL}'
      EXTERNAL_ACCESS_INTEGRATIONS = (GRAPH_API_ACCESS_INTEGRATION, PYPI_ACCESS_INTEGRATION)
      SECRETS = ('graph_secret' = {APP_GRAPH_SECRET})
    """
else:
    create_stmt += f"""
      RUNTIME_NAME = 'SYSTEM$WAREHOUSE_RUNTIME'
      EXTERNAL_ACCESS_INTEGRATIONS = (GRAPH_API_ACCESS_INTEGRATION)
      SECRETS = ('graph_secret' = {APP_GRAPH_SECRET})
    """

session.sql(create_stmt).collect()

print(f"Streamlit app deployed: {STREAMLIT_APP_NAME}")
print(f"  Warehouse: {APP_QUERY_WAREHOUSE}")
print(f"  Runtime:   {'container (' + APP_COMPUTE_POOL + ')' if APP_COMPUTE_POOL else 'warehouse'}")
print(f"  Manifest:  {manifest_to_stage} (removed {manifest_to_remove} if present)")
print(f"  Graph secret bound: {APP_GRAPH_SECRET}")
print(f"  Grant USAGE ON STREAMLIT {STREAMLIT_APP_NAME} TO ROLE LEX_USERS has already been run above.")

## Schema migrations (safe to re-run any time)

New columns added to `RAW_DOCUMENTS`/`DOCUMENT_INDEX`/`PROJECTS`/LEX's own
contract tables after a project was first created won't retroactively
appear in its already-existing schema — the project-creation and
contract-tables cells above only run `CREATE ... IF NOT EXISTS`, which
does nothing to a table that already exists in an older shape. This cell
is the running list of forward-only, idempotent `ALTER TABLE ... ADD
COLUMN IF NOT EXISTS` statements for LEX's schema specifically, so picking
up a schema change is "re-run this cell" rather than a manual one-off
`ALTER TABLE` typed into a worksheet — the same discipline `ORG_MM_CHAT`'s
own equivalent cell uses. A project provisioned for the first time with
the current version of this notebook already has every column from the
`CREATE TABLE` statements above and these are no-ops for it; they only
matter for a LEX project that existed before this cell's entries were
added.

In [ ]:
qualified_schema = f"{proj_row[0]['DATA_DATABASE']}.{proj_row[0]['DATA_SCHEMA']}"

migrations = [
    # (contract-lookup UX) OVERVIEW_SUMMARY: the template's "Executive
    # Assessment" narrative, shown at the top of Contract Lookup and in
    # the .docx export — see contract_extraction.generate_contract_overview.
    f"ALTER TABLE {qualified_schema}.CONTRACT_REGISTER ADD COLUMN IF NOT EXISTS OVERVIEW_SUMMARY VARCHAR(4000)",
    f"ALTER TABLE {qualified_schema}.CONTRACT_REGISTER ADD COLUMN IF NOT EXISTS OVERVIEW_GENERATED_AT TIMESTAMP_NTZ",
    # (Contract Workspace Summary Template adoption) RECOMMENDED_ACTIONS /
    # CLASSIFICATION_SCORECARD: the template's "Recommended Actions" bullet
    # list (JSON array) and "Consolidated Procurement Assessment" scorecard
    # (JSON object) — see contract_extraction.generate_recommended_actions /
    # generate_classification_scorecard.
    f"ALTER TABLE {qualified_schema}.CONTRACT_REGISTER ADD COLUMN IF NOT EXISTS RECOMMENDED_ACTIONS VARIANT",
    f"ALTER TABLE {qualified_schema}.CONTRACT_REGISTER ADD COLUMN IF NOT EXISTS CLASSIFICATION_SCORECARD VARIANT",
    # (citation viewer) HIGHLIGHT_PHRASE: the short exact phrase the
    # citation panel highlights/searches for, verified as a substring of
    # SOURCE_QUOTE at extraction time — see contract_extraction.py's
    # _extract_highlight_phrase.
    f"ALTER TABLE {qualified_schema}.CONTRACT_FIELD_EXTRACTS ADD COLUMN IF NOT EXISTS HIGHLIGHT_PHRASE VARCHAR(500)",
    # SOURCE_QUOTE widened 2000 -> 4000: it now holds the cited section's
    # full excerpt (for the citation panel's exact-text view), not just a
    # short snippet. COLLATE 'en-ci' is required alongside SET DATA TYPE on
    # this account even for a pure length widen with no other type change —
    # ALTER COLUMN must match the existing column's collation exactly, and
    # this account applies 'en-ci' as the default VARCHAR collation (the
    # same gotcha the ORG_MM_CHAT notebook hit widening NODE_SUMMARY).
    f"""ALTER TABLE {qualified_schema}.CONTRACT_FIELD_EXTRACTS
        ALTER COLUMN SOURCE_QUOTE SET DATA TYPE VARCHAR(4000) COLLATE 'en-ci'""",
]

for stmt in migrations:
    session.sql(stmt).collect()
    print(f"OK  {stmt}")

## If the app still won't load
Snowsight's "Something went wrong" card only shows the top-level exception,
not a traceback. Run the cell below in *this* notebook (same Python
environment the app runs in) to import every current app dependency one at
a time and print a full traceback for whichever one actually fails.

In [ ]:
import sys, traceback

for p in ("../python", "../streamlit"):
    if p not in sys.path:
        sys.path.insert(0, p)

# Same import order Chat.py exercises, split out module by module so
# whichever one fails prints its own full traceback instead of one opaque
# top-level error.
modules_to_test = [
    "requests",
    "pandas",
    "config",
    "snowflake_session",
    "query_engine",
    "contract_linking",
    "contract_extraction",
    "required_contracts",
    "citation_viewer",
    "citation_panel_ui",
    "docx_report",
    "docx",
    "utils.cortex_client",
    "utils.graph_client",
    "utils.sql_utils",
    "utils.sql_script",
    "utils.logging_utils",
    "ingestion.xlsx_parser",
    "ingestion.file_ingest",
    "ingestion.sharepoint_ingest",
    "ingestion.index_builder",
]

for m in modules_to_test:
    try:
        __import__(m)
        print(f"OK    {m}")
    except Exception:
        print(f"FAILED {m}")
        traceback.print_exc()
        print()

print("\nPython:", sys.version)

## Debug: test JSON parsing on one document

If **Index new/unindexed documents** in the app fails with a Cortex
JSON-parsing error (`complete_json` in `python/utils/cortex_client.py`),
use this cell to reproduce it directly against one live document — seconds,
not a full Streamlit redeploy-and-click cycle. It calls the real
`complete`/`complete_json` functions from this repo, so a fix verified
here is testing the actual code path the app uses.

Makes two live Cortex calls — fine for interactive debugging, but don't
loop this over many documents; use the app's own indexing for that.

In [ ]:
import sys
if "../python" not in sys.path:
    sys.path.insert(0, "../python")

from config import load_project
from utils.cortex_client import complete, complete_json
from ingestion.index_builder import PROMPTS

DEBUG_PROJECT_CODE = PROJECT_CODE  # from the project-creation cell above
DEBUG_DOC_ID = None  # a specific RAW_DOCUMENTS.DOC_ID, or None for "first document"

project = load_project(session, DEBUG_PROJECT_CODE)
schema = project.qualified_schema
prompt_template = PROMPTS.get(project.segmentation_profile, PROMPTS["GENERIC"])

where = "DOC_ID = ?" if DEBUG_DOC_ID else "1=1"
params = [DEBUG_DOC_ID] if DEBUG_DOC_ID else []
doc = session.sql(
    f"SELECT DOC_ID, FILE_NAME, RAW_TEXT FROM {schema}.RAW_DOCUMENTS "
    f"WHERE {where} ORDER BY DOC_ID LIMIT 1",
    params=params,
).collect()[0]

print(f"Testing DOC_ID={doc['DOC_ID']} FILE_NAME={doc['FILE_NAME']!r}")
# Uses only the first chunk's worth of text — matches what
# index_builder._index_one_document sends in its first (and, for most
# documents, only) indexing call; see that function for the full chunked
# path used on documents longer than project.max_document_chars.
text = doc["RAW_TEXT"][: project.max_document_chars]
prompt = prompt_template.format(text=text, granularity_instruction="")

print("\n--- Step 1: raw Cortex response (full text) ---")
raw = complete(session, project.active_model, prompt)
print(raw)

print("\n--- Step 2: complete_json() result ---")
try:
    result = complete_json(session, project.active_model, prompt)
    print("Parsed OK. Keys:", list(result.keys()))
    print("document_summary:", result.get("document_summary", "")[:200])
    print("sections found:", len(result.get("sections", [])))
except Exception as e:
    print(f"FAILED: {type(e).__name__}: {e}")

## Next step
Open the Streamlit app and, on **Data Sources**: (1) upload the Required
Contracts Register workbook (the list of CW numbers currently in scope —
2 today, growing toward 8 for Build/validation) under its own tab, then
(2) ingest each contract's signed/executed PDF (rarely DOCX). Once a
contract and any of its variations/extensions are ingested, go to
**Contract Register** to link them into one family and run extraction —
after that, look the contract up on **Contract Lookup** (the app's landing
page) to see its standard questions answered laid out the same way as the
Contract Workspace Summary Template, click through to the cited passage in
the original document, and download the matching .docx summary. No further
notebook steps are needed for day-to-day use — this notebook is for
provisioning and redeploys only.